In [1]:
import cmdstanpy as csp
import numpy as np
import pandas as pd
import logging
import warnings
from tqdm import tqdm
import arviz as az

In [2]:
# warnings.simplefilter(action='ignore', category=FutureWarning)
# warnings.filterwarnings( "ignore", module = "plotnine\..*" )
csp.utils.get_logger().setLevel(logging.ERROR)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


def rating_csv_to_dict(file):
    df = pd.read_csv(file, comment = '#')
    rater = df['rater'].to_list()
    item = df['item'].to_list()
    rating = df['rating'].to_list()
    I = int(np.max(item))
    J = int(np.max(rater))
    N = int(len(rater))
    data = { 'I': I, 'J': J, 'N': N,
             'item': item, 'rater': rater, 'rating': rating }
    return data

def sample(stan_file, data, init = {}):
    model = csp.CmdStanModel(stan_file = stan_file)
    sample = model.sample(data = data, inits = init,
                          iter_warmup=1000, iter_sampling=1000,
                          chains = 2, parallel_chains = 4,
                          show_console = True, show_progress=True,
                          refresh = 100,
                          seed = 925845)
    return sample

def pathfind(stan_file, data, init = {}):
    model = csp.CmdStanModel(stan_file = stan_file)
    fit = model.pathfinder(data = data, inits = init, show_console=True, refresh = 5)
    return fit

def min_p_twosided(ps):
    return np.fmin(np.min(ps), 1 - np.max(ps)) / 2

In [3]:
data_file = 'rte.csv'
data_path = '../data/' + data_file
data = rating_csv_to_dict(data_path)
print(data)
init = {
    'pi': 0.2,
    'alpha_acc_scalar': 2,
    'alpha_sens_scalar': 1,
    'alpha_spec_scalar': 2,
    'alpha_acc': np.full(data['J'], 2),
    'alpha_sens': np.full(data['J'], 1),
    'alpha_spec': np.full(data['J'], 2),
    'beta': np.full(data['I'], 0),
    'delta': np.full(data['I'], 1),
    'lambda': np.full(data['I'], 0.5)
}         
J = data['J']
    
rater_labels = [f"rater_sim[{i}]" for i in range(1, J)]
rater_lt_labels = [f"rater_sim_lt_data[{i}]" for i in range(1, J)]
votes_labels = [f"votes_sim[{i}]" for i in range(1, J + 1)]
votes_lt_labels = [f"votes_sim_lt_data[{i}]" for i in range(1, J + 1)]


models = ['a', 'ab', 'abc', 'abcd', 'abcde', 'abce', 'abd', 'abde', 'ac', 'acd', 'ad', 'bc', 'bcd', 'bd', 'c', 'cd', 'd', 'full']
rows = []

{'I': 800, 'J': 164, 'N': 8000, 'item': [30, 30, 30, 30, 30, 30, 30, 30, 30, 30, 177, 177, 177, 177, 177, 177, 177, 177, 177, 177, 182, 182, 182, 182, 182, 182, 182, 182, 182, 182, 584, 584, 584, 584, 584, 584, 584, 584, 584, 584, 15, 15, 15, 15, 15, 15, 15, 15, 15, 15, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 10, 10, 10, 10, 10, 10, 10, 10, 10, 10, 44, 44, 44, 44, 44, 44, 44, 44, 44, 44, 778, 778, 778, 778, 778, 778, 778, 778, 778, 778, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 180, 180, 180, 180, 180, 180, 180, 180, 180, 180, 184, 184, 184, 184, 184, 184, 184, 184, 184, 184, 34, 34, 34, 34, 34, 34, 34, 34, 34, 34, 772, 772, 772, 772, 772, 772, 772, 772, 772, 772, 610, 610, 610, 610, 610, 610, 610, 610, 610, 610, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 57, 57, 57, 57, 57, 57, 57, 57, 57, 57, 60, 60, 60, 60, 60, 60, 60, 60, 60, 60, 178, 178, 178, 178, 178, 178, 178, 178, 178, 178, 611, 611, 611, 611, 611, 611, 611, 611, 611, 611, 766, 766, 766, 766, 766, 766, 766, 766, 766, 766, 586, 586, 586, 586,

In [4]:
for model in models:
    print(f"***** {model = }")
    draws = sample('../stan/' + model + '.stan', data, init)
    log_lik_pointwise = draws.stan_variable('log_lik_pointwise')
    num_samples, num_observations = log_lik_pointwise.shape
    
    # Coordinates for ArviZ
    coords = {
        "sample": np.arange(num_samples),
        "observation": np.arange(num_observations)
    }
    
    # Correct dimensions for ArviZ
    log_lik_dims = {"log_lik": ["sample", "observation"]}
    
    # Convert the CmdStanPy output to ArviZ InferenceData
    log_lik_idata = az.from_cmdstanpy(
        posterior=draws,
        log_likelihood="log_lik_pointwise",
        coords=coords,
        dims={"log_likelihood": log_lik_dims}
    )
    loo_results = az.loo(log_lik_idata)
    
    # Collecting the LOO statistics
    row = {
        'model': [model],
        'elpd_loo': loo_results['elpd_loo'],  # Access the elpd_loo estimate
        'loo_se': loo_results['se'],          # Access the standard error (SE) of elpd_loo
        'p_loo': loo_results['p_loo']         # Access the effective number of parameters
    }
    
    rows.append(pd.DataFrame([row]))

# Save all the LOO results into a CSV file
results_df = pd.concat(rows)
results_df.to_csv('loo_results_rte.csv', index=False, sep=',', encoding='utf-8')

***** model = 'a'
Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 1000 (Default)
Chain [1] num_warmup = 1000 (Default)
Chain [1] save_warmup = 0 (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = 1 (Default)
Chain [1] gamma = 0.050000 (Default)
Chain [1] delta = 0.800000 (Default)
Chain [1] kappa = 0.750000 (Default)
Chain [1] t0 = 10.000000 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = 0 (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 10 (Default)
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1.000000 (Default)
Chain [1] stepsize_jitter = 0.000000 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /tmp/tmpsqzbnqfg/ojpmj9bm.json
Chain [1] init = /tmp/tmpsqzbnqfg/

/home/seong/env_jupyter/lib/python3.10/site-packages/arviz/stats/stats.py:805: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.7 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


***** model = 'abc'
Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 1000 (Default)
Chain [1] num_warmup = 1000 (Default)
Chain [1] save_warmup = 0 (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = 1 (Default)
Chain [1] gamma = 0.050000 (Default)
Chain [1] delta = 0.800000 (Default)
Chain [1] kappa = 0.750000 (Default)
Chain [1] t0 = 10.000000 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = 0 (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 10 (Default)
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1.000000 (Default)
Chain [1] stepsize_jitter = 0.000000 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /tmp/tmpsqzbnqfg/fiibu_1i.json
Chain [1] init = /tmp/tmpsqzbnqf

/home/seong/env_jupyter/lib/python3.10/site-packages/arviz/stats/stats.py:805: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.7 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


***** model = 'cd'
Chain [1] method = sample (Default)
Chain [1] sample
Chain [1] num_samples = 1000 (Default)
Chain [1] num_warmup = 1000 (Default)
Chain [1] save_warmup = 0 (Default)
Chain [1] thin = 1 (Default)
Chain [1] adapt
Chain [1] engaged = 1 (Default)
Chain [1] gamma = 0.050000 (Default)
Chain [1] delta = 0.800000 (Default)
Chain [1] kappa = 0.750000 (Default)
Chain [1] t0 = 10.000000 (Default)
Chain [1] init_buffer = 75 (Default)
Chain [1] term_buffer = 50 (Default)
Chain [1] window = 25 (Default)
Chain [1] save_metric = 0 (Default)
Chain [1] algorithm = hmc (Default)
Chain [1] hmc
Chain [1] engine = nuts (Default)
Chain [1] nuts
Chain [1] max_depth = 10 (Default)
Chain [1] metric = diag_e (Default)
Chain [1] metric_file =  (Default)
Chain [1] stepsize = 1.000000 (Default)
Chain [1] stepsize_jitter = 0.000000 (Default)
Chain [1] num_chains = 1 (Default)
Chain [1] id = 1 (Default)
Chain [1] data
Chain [1] file = /tmp/tmpsqzbnqfg/3p6sj7aw.json
Chain [1] init = /tmp/tmpsqzbnqfg

In [141]:
row = {
        'model': [model],
        'elpd_loo': loo_results['elpd_loo'],  # Access the elpd_loo estimate
        'loo_se': loo_results['se'],          # Access the standard error (SE) of elpd_loo
        'p_loo': loo_results['p_loo']         # Access the effective number of parameters
    }

In [142]:
row

{'model': ['full'],
 'elpd_loo': -8678.884741695434,
 'loo_se': 78.45873176610756,
 'p_loo': 476.21819912632054}

In [122]:
log_lik_pointwise = draws.stan_variable('log_lik_pointwise')
num_samples, num_observations = log_lik_pointwise.shape

# Coordinates for ArviZ
coords = {
    "sample": np.arange(num_samples),
    "observation": np.arange(num_observations)
}

# Correct dimensions for ArviZ
log_lik_dims = {"log_lik": ["sample", "observation"]}

# Convert the CmdStanPy output to ArviZ InferenceData
log_lik_idata = az.from_cmdstanpy(
    posterior=draws,
    log_likelihood="log_lik_pointwise",
    coords=coords,
    dims={"log_likelihood": log_lik_dims}
)

In [123]:
loo_results = az.loo(log_lik_idata)
print(loo_results)

/home/seong/env_jupyter/lib/python3.10/site-packages/arviz/stats/stats.py:805: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.7 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


Computed from 2000 posterior samples and 19345 observations log-likelihood matrix.

         Estimate       SE
elpd_loo -8678.88    78.46
p_loo      476.22        -

There has been a warning during the calculation. Please check the results.
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.5]   (good)     18979   98.1%
 (0.5, 0.7]   (ok)         363    1.9%
   (0.7, 1]   (bad)          3    0.0%
   (1, Inf)   (very bad)     0    0.0%



In [113]:
# model a
log_lik_pointwise = draws.stan_variable('log_lik_pointwise')
num_samples, num_observations = log_lik_pointwise.shape

# Coordinates for ArviZ
coords = {
    "sample": np.arange(num_samples),
    "observation": np.arange(num_observations)
}

# Correct dimensions for ArviZ
log_lik_dims = {"log_lik": ["sample", "observation"]}

# Convert the CmdStanPy output to ArviZ InferenceData
log_lik_idata = az.from_cmdstanpy(
    posterior=draws,
    log_likelihood="log_lik_pointwise",
    coords=coords,
    dims={"log_likelihood": log_lik_dims}
)

In [114]:
loo_results = az.loo(log_lik_idata)
print(loo_results)

/home/seong/env_jupyter/lib/python3.10/site-packages/arviz/stats/stats.py:805: UserWarning: Estimated shape parameter of Pareto distribution is greater than 0.7 for one or more samples. You should consider using a more robust model, this is because importance sampling is less likely to work well if the marginal posterior and LOO posterior are very different. This is more likely to happen with a non-robust model and highly influential observations.
  warnings.warn(


Computed from 2000 posterior samples and 19345 observations log-likelihood matrix.

         Estimate       SE
elpd_loo -8696.63    79.99
p_loo      403.19        -

There has been a warning during the calculation. Please check the results.
------

Pareto k diagnostic values:
                         Count   Pct.
(-Inf, 0.5]   (good)     19272   99.6%
 (0.5, 0.7]   (ok)          71    0.4%
   (0.7, 1]   (bad)          2    0.0%
   (1, Inf)   (very bad)     0    0.0%

